# 01 — Exploration des données (EDA)

**Projet** : Segmentation client d'un e-commerçant (RFM étendu) avec scikit-learn
**Cas métier** : Découvrir **sans étiquette préalable** des groupes de clients homogènes et actionnables à partir de leur comportement d'achat et d'engagement sur douze mois glissants, choisir le nombre de groupes de façon argumentée (et non arbitraire), profiler chaque groupe, puis affecter tout nouveau client au groupe le plus proche avec un niveau de confiance.
**Jeu de données** : `retail_customer_base` — Base clients e-commerce et comportement d'achat sur 12 mois

> Population de clients d'un e-commerçant (mode et maison) avec leurs agrégats comportementaux sur douze mois glissants. Les données sont générées à partir de **six profils latents** (VIP fidèle, chasseur de promotions, acheteur occasionnel, dormeur, nouveau curieux, client insatisfait) : chaque profil définit des distributions propres (fréquence, panier, part promotionnelle, taux de retour, engagement), puis un bruit individuel et des chevauchements volontaires sont injectés. La structure est donc réelle mais **non triviale** : les groupes se recouvrent partiellement, ce qui rend le choix du nombre de clusters et la lecture de la silhouette authentiquement difficiles. Deux colonnes de métadonnées (segment latent et churn observé à 90 jours) permettent de mesurer la qualité de la segmentation sans jamais entrer dans les features.

## Objectifs pédagogiques

1. Charger un jeu de données tabulaire et en établir le profil (types, manquants, doublons).
1. Lire une distribution : détecter déséquilibre, outliers et colinéarité **avant** de modéliser.
1. Relier chaque observation statistique à une conséquence métier ou de modélisation.
1. Produire les figures qui serviront de référence dans les notebooks suivants.

**Objectifs transverses du dépôt**

- Construire un pipeline non supervisé sans fuite : les colonnes de diagnostic sont exclues des features par configuration.
- Choisir le nombre de groupes par triangulation (silhouette, Davies-Bouldin, coude d'inertie, taille minimale).
- Comprendre l'effet de l'échelle et des queues de distribution sur une distance euclidienne (log, winsorising, standardisation).

## 0. Environnement

Toute la configuration vient de **Hydra** (`conf/`) : aucune valeur métier n'est codée en dur
dans ce notebook. Si `data/raw` est vide, le générateur synthétique du projet prend le relais
(voir `make data`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.25)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

**Pourquoi ce bloc d'initialisation**

- `CONFIG` est l'objet **Pydantic** validé : une clé incohérente échoue ici, pas en production.
- Les notebooks travaillent sur un échantillon réduit pour rester rapides ; `make train` utilise `data.n_samples` complet.
- `NB_PATHS` isole les écritures du notebook dans `outputs/notebooks`.

## 1. Chargement et premier contact

On ne regarde jamais un dataset sans vérifier trois choses : sa **forme** (lignes x colonnes),
ses **types** (un numérique lu comme texte casse tout) et ses **premières lignes** (les valeurs
ont-elles du sens métier ?).

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

**Ce qu'il faut retenir**

- Le contrat `retail_customer_base` est documenté dans `data/README.md` : chaque colonne y a une signification métier.
- Les identifiants et horodatages ne sont **pas** des features : ils servent à tracer et à splitter.

In [ ]:
from src.data.schemas import describe_schema, validation_report

# Types déclarés (contrat Pandera) vs types réellement lus : toute divergence est un signal.
contract = describe_schema("raw")
observed = pd.DataFrame({"dtype_lu": {str(k): str(v) for k, v in raw.dtypes.items()}})
contract.join(observed)[["dtype", "dtype_lu", "nullable", "unique", "checks"]]

**Ce qu'il faut retenir**

- La colonne `dtype` vient du **contrat**, `dtype_lu` de la source : elles doivent correspondre.
- Les `checks` (bornes, valeurs autorisées) sont la mémoire des règles métier — ils seront testés au notebook 02.

In [ ]:
report = validation_report(raw)
summary = pd.DataFrame(
    {
        "indicateur": [
            "lignes",
            "colonnes",
            "cellules manquantes",
            "taux de manquants",
            "mémoire (Ko)",
        ],
        "valeur": [
            report["n_rows"],
            report["n_columns"],
            report["missing_cells"],
            f"{report['missing_rate']:.2%}",
            round(report["memory_kb"], 1),
        ],
    }
)
summary

**Ce qu'il faut retenir**

- Un taux de manquants global faible peut cacher une colonne très incomplète : regarder **par colonne**.
- La mémoire indique si le dataset tient en RAM (sinon : pyarrow, chunking ou échantillonnage).

## 2. Valeurs manquantes

Où, combien, et surtout : **manquant au hasard ou pas** ? Un manquant informatif (ex. score de satisfaction non renseigné par les clients mécontents) est un signal, pas seulement un problème technique.

In [ ]:
missing = raw.isna().sum()
missing_frame = (
    pd.DataFrame({"manquants": missing, "taux": (missing / len(raw)).round(4)})
    .loc[lambda frame: frame["manquants"] > 0]
    .sort_values("manquants", ascending=False)
)
missing_frame

In [ ]:
if missing_frame.empty:
    print("Aucune valeur manquante dans cet échantillon.")
else:
    fig, axis = plt.subplots(figsize=(7.5, 0.55 * len(missing_frame) + 1.6))
    axis.barh(missing_frame.index[::-1], missing_frame["taux"][::-1] * 100, color="#d1495b")
    axis.set_xlabel("Cellules manquantes (%)")
    axis.set_title("Valeurs manquantes par colonne")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- L'imputation doit être **apprise sur le train** (moyenne/médiane/constante) puis appliquée aux autres splits.
- Ajouter un indicateur binaire « valeur manquante » est souvent rentable quand le manquant est informatif.
- Notes du générateur : Valeurs manquantes volontaires sur `nps_score` (~8 %) et `avg_basket_eur` (clients sans commande).; Outliers légitimes (~1 %) : clients d'exception (gros volumes, paniers très élevés)..

## 3. Distributions numériques

In [ ]:
numeric_columns = [column for column in raw.columns if pd.api.types.is_numeric_dtype(raw[column])]
numeric_columns = [column for column in numeric_columns if column != CONFIG.data.target]

n_plots = len(numeric_columns)
n_cols = 3
n_rows = int(np.ceil(n_plots / n_cols)) if n_plots else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 2.7 * n_rows))
for axis, column in zip(np.atleast_1d(axes).ravel(), numeric_columns, strict=False):
    raw[column].hist(bins=30, ax=axis, color="#005f73", edgecolor="white")
    axis.set_title(column, fontsize=9)
    axis.tick_params(labelsize=7)
for axis in np.atleast_1d(axes).ravel()[len(numeric_columns) :]:
    axis.axis("off")
fig.suptitle("Distributions des variables numériques", y=1.005)
fig.tight_layout()
plt.show()

In [ ]:
raw[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

**Ce qu'il faut retenir**

- Une distribution très asymétrique (max ≫ p99) justifie un **winsorising** ou un `log1p` plutôt qu'une suppression d'outliers.
- Des échelles hétérogènes (euros, Go, unités) imposent un **scaling** pour les modèles sensibles à la distance (SVM, k-NN, réseaux).
- Comparer `mean` et `50%` : un écart important signale une queue lourde.

## 4. Variables catégorielles

In [ ]:
categorical_columns = [
    column
    for column in raw.columns
    if not pd.api.types.is_numeric_dtype(raw[column])
    and column not in [*CONFIG.data.drop_columns, str(CONFIG.data.target)]
]

for column in categorical_columns:
    counts = raw[column].astype(str).value_counts()
    print(f"--- {column} ({len(counts)} modalités) ---")
    print((counts / len(raw)).map("{:.1%}".format).to_string())

**Ce qu'il faut retenir**

- Une modalité ultra-rare (< 1 %) doit être regroupée dans un bucket `rare` : sinon l'encodage one-hot crée des colonnes quasi vides et instables.
- Une cardinalité élevée (identifiants, codes postaux) appelle un **target encoding** régularisé plutôt qu'un one-hot.

## 5. Pas de cible : quelle structure cherche-t-on ?

Il n'y a **aucune variable à prédire** ici. L'exploration ne consiste donc pas à expliquer une
cible, mais à répondre à trois questions qui conditionnent toute la suite :

1. **échelle** — les variables sont-elles comparables ? Une distance euclidienne sur des euros et
   des parts (0-1) est dominée par les euros : le clustering serait une segmentation par chiffre
   d'affaires, rien d'autre ;
2. **structure** — existe-t-il une géométrie exploitable (groupes, densités) ou un continuum ?
   L'ACP donne une première réponse honnête : si deux axes expliquent tout, la segmentation sera
   une découpe de plan ;
3. **diagnostic** — de quoi disposera-t-on pour juger la segmentation *après* coup ? Deux colonnes
   de métadonnées existent dans ce jeu synthétique (profil latent injecté, churn observé à 90
   jours). Elles sont **exclues des features** par `drop_columns` : elles servent à valider la
   méthode, pas à l'entraîner.

In [ ]:
feature_columns = [column for column in raw.columns if column not in set(CONFIG.data.drop_columns)]
numeric_columns = [
    column for column in feature_columns if pd.api.types.is_numeric_dtype(raw[column])
]

scale = pd.DataFrame(
    {
        "min": raw[numeric_columns].min(),
        "médiane": raw[numeric_columns].median(),
        "max": raw[numeric_columns].max(),
        "écart-type": raw[numeric_columns].std(),
    }
)
print("Rapport max/écart-type — plus il est grand, plus la variable domine une distance brute :")
scale.assign(**{"max / écart-type": (scale["max"] / scale["écart-type"]).round(1)}).sort_values(
    "max / écart-type", ascending=False
).round(2)

**Ce qu'il faut retenir**

- Sans standardisation, `revenue_12m_eur` (milliers d'euros) écrase `discount_share` (0 à 1) : la distance euclidienne ne verrait que le chiffre d'affaires.
- C'est la première cause d'échec d'un clustering en production — et elle est invisible si l'on regarde seulement la silhouette (elle aussi calculée sur des distances).
- Les queues lourdes (revenus, sessions) appellent un `log1p` **avant** le scaling : c'est exactement ce que fait `conf/preprocessing/default.yaml`.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ACP sur les features numériques standardisées : c'est l'espace dans lequel le clustering
# travaillera (à l'encodage des catégorielles près).
matrix = (
    raw[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(raw[numeric_columns].median(numeric_only=True))
)
scaled = StandardScaler().fit_transform(matrix.to_numpy(dtype="float64"))

pca = PCA(n_components=4, random_state=CONFIG.seed)
projected = pca.fit_transform(scaled)

variance = pd.DataFrame(
    {
        "axe": [f"PC{index}" for index in range(1, len(pca.explained_variance_ratio_) + 1)],
        "variance_expliquée": pca.explained_variance_ratio_.round(4),
        "cumul": pca.explained_variance_ratio_.cumsum().round(4),
    }
)
print(variance.to_string(index=False))

diagnostic_column = "latent_segment" if "latent_segment" in raw.columns else None
map_frame = pd.DataFrame(projected[:, :2], columns=["x", "y"])
fig, axis = plt.subplots(figsize=(8.4, 5.6))
if diagnostic_column:
    # Coloration par le profil latent INJECTÉ PAR LE GÉNÉRATEUR : diagnostic pédagogique
    # uniquement. En production, cette colonne n'existe pas — le clustering est aveugle.
    map_frame["profil"] = raw[diagnostic_column].to_numpy()
    for level, group in map_frame.groupby("profil", observed=True):
        axis.scatter(group["x"], group["y"], s=13, alpha=0.55, label=str(level), edgecolors="none")
    axis.legend(title="profil latent (diagnostic)", fontsize=8, title_fontsize=8, markerscale=1.6)
else:
    axis.scatter(
        map_frame["x"], map_frame["y"], s=13, alpha=0.55, color="#005f73", edgecolors="none"
    )
axis.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axis.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axis.set_title("Structure du nuage de clients (ACP 2D)")
fig.tight_layout()
plt.show()

loadings = pd.DataFrame(pca.components_[0:2].T, index=numeric_columns, columns=["PC1", "PC2"])
loadings.reindex(loadings.abs().sum(axis=1).sort_values(ascending=False).index).head(8).round(3)

**Ce qu'il faut retenir**

- Si les profils latents se séparent nettement sur les deux premiers axes, la structure est apprenable ; s'ils se mélangent, aucune méthode ne les séparera proprement — et c'est le cas ici par construction (chevauchement volontaire).
- Les loadings disent **quoi** portent les axes : un PC1 dominé par le chiffre d'affaires et la fréquence est un axe de valeur, un PC2 dominé par la part promotionnelle est un axe de comportement.
- Une variance expliquée faible sur les deux premiers axes n'est pas un échec : la structure vit alors en dimension supérieure, et la carte 2D sous-estime la séparation réelle.

In [ ]:
diagnostic_columns = [
    column for column in ("latent_segment", "churned_next_90d") if column in raw.columns
]

if not diagnostic_columns:
    print("Aucune colonne de diagnostic dans ce jeu de données.")
elif "latent_segment" in raw and "churned_next_90d" in raw:
    external = raw.groupby("latent_segment", observed=True).agg(
        clients=("churned_next_90d", "size"),
        churn_90j=("churned_next_90d", "mean"),
    )
    external["part"] = external["clients"] / len(raw)
    print("Validité externe disponible : le churn observé à 90 jours, par profil latent")
    display(external.round(3).sort_values("churn_90j", ascending=False))
    spread = external["churn_90j"].max() - external["churn_90j"].min()
    print(f"écart de churn entre profils : {spread:.1%}")
    print("Répartition des profils latents (parts attendues dans la population) :")
    display((raw["latent_segment"].value_counts(normalize=True) * 100).round(1).to_frame("%"))
else:
    print("Colonnes de diagnostic présentes :", diagnostic_columns)

**Ce qu'il faut retenir**

- Ces deux colonnes sont des **métadonnées** : `drop_columns` les exclut de la matrice de features. Elles ne servent qu'à juger la segmentation une fois produite.
- Le churn observé à 90 jours est la mesure de **validité externe** : une segmentation dont les groupes ont tous le même taux de churn est statistiquement propre et commercialement inutile.
- L'accord avec le profil latent (ARI, NMI) est un diagnostic pédagogique : en production, personne ne connaît la « vraie » segmentation — c'est pourquoi les critères internes et métier priment.

## 6. Colinéarité et structure

In [ ]:
correlation = raw[numeric_columns].corr(numeric_only=True)
fig, axis = plt.subplots(figsize=(6.6, 5.4))
image = axis.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(
    range(len(correlation.columns)), correlation.columns, rotation=45, ha="right", fontsize=7
)
axis.set_yticks(range(len(correlation.index)), correlation.index, fontsize=7)
for row in range(correlation.shape[0]):
    for column in range(correlation.shape[1]):
        value = correlation.iloc[row, column]
        axis.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=6,
            color="black" if abs(value) < 0.6 else "white",
        )
fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
axis.set_title("Corrélations de Pearson (variables numériques)")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Deux features corrélées à > 0.9 n'apportent presque rien ensemble : en garder une simplifie le modèle et son explication.
- Les arbres sont robustes à la colinéarité ; les modèles linéaires/régularisés voient leurs coefficients devenir instables.
- La corrélation ne capture pas les relations **non linéaires** : la vérifier par des graphes cible vs feature.

In [ ]:
# Outliers : comptage par la règle de l'IQR (1.5 x écart interquartile).
rows = []
for column in numeric_columns:
    series = raw[column].dropna()
    if series.empty:
        continue
    low, high = series.quantile([0.25, 0.75])
    iqr = high - low
    outliers = int(((series < low - 1.5 * iqr) | (series > high + 1.5 * iqr)).sum())
    rows.append(
        {"colonne": column, "outliers_iqr": outliers, "part": outliers / max(len(series), 1)}
    )
outlier_frame = pd.DataFrame(rows).sort_values("outliers_iqr", ascending=False)
outlier_frame.head(8).round(4)

**Ce qu'il faut retenir**

- La règle IQR **signale**, elle ne tranche pas : un outlier peut être un client légitime (grand compte, pic saisonnier).
- Le winsorising (clip aux quantiles 1-99 %) conserve les lignes et les labels, contrairement à la suppression.

In [ ]:
# Intégrité : unicité de la clé et doublons complets.
key = CONFIG.data.id_column
duplicates = int(raw.duplicated().sum())
key_duplicates = int(raw[key].duplicated().sum()) if key and key in raw.columns else 0
print(f"doublons complets            : {duplicates}")
print(f"doublons sur la clé '{key}' : {key_duplicates}")
unique_keys = raw[key].nunique() if key in raw.columns else "n/a"
print(f"identifiants uniques         : {unique_keys} / {len(raw)}")

## 7. Synthèse de l'exploration

**Lectures clés de ce jeu de données**

- Il n'y a **pas de cible** : la qualité se juge sur des critères internes (silhouette, Davies-Bouldin, Calinski-Harabasz), externes (accord avec le segment latent, écart de churn) et **métier** (taille minimale, stabilité, actionnabilité).
- La silhouette est très sensible à l'échelle des variables : `revenue_12m_eur` (euros) et `discount_share` (0-1) ne peuvent pas cohabiter sans standardisation. C'est la première raison d'échec d'un clustering en production.
- `revenue_12m_eur`, `web_sessions_12m` et `orders_12m` sont fortement asymétriques (queue droite) : un passage en log et un winsorising stabilisent les centroïdes, sinon quelques clients d'exception tirent un groupe entier.
- `avg_basket_eur` est quasi redondant avec `revenue_12m_eur / orders_12m` : la redondance gonfle artificiellement la séparation et biaise la silhouette au profit des variables les plus lourdes.
- Les groupes se **chevauchent** volontairement : un acheteur occasionnel qui profite d'une promotion ressemble à un chasseur de promo. Une silhouette de 0.25-0.40 est donc un résultat honnête, pas un échec.
- `recency_days` est bimodal (actifs vs dormeurs) : c'est la variable la plus discriminante, mais elle ne suffit pas — deux clients à 300 jours d'inactivité peuvent être un dormeur et un VIP en pause.
- `discount_share` et `newsletter_opens_12m` séparent le profil promotionnel du profil fidèle plein tarif : c'est le levier d'arbitrage du budget remises.
- `return_rate` et `support_tickets_12m` signalent l'insatisfaction avant le churn : un groupe à fort taux de retour est un gisement de rétention, pas un groupe à solliciter davantage.
- `nps_score` comporte ~8 % de manquants (biais de non-réponse : les détracteurs répondent davantage) ; l'imputer par la médiane masque ce biais, un indicateur de manquant est préférable.
- `loyalty_tier` est **calculé par le métier** à partir du chiffre d'affaires : l'utiliser comme feature introduit une circularité (on redécouvre le palier). À documenter, voire à exclure.
- `region` et `acquisition_channel` sont des proxies socio-économiques : une segmentation qui sépare principalement par région poserait un problème d'équité et de conformité.
- Le nombre de groupes est un **choix** : k trop petit fusionne des comportements distincts, k trop grand produit des micro-groupes inexploitables par le CRM (coût de campagne, lisibilité).

### Décisions de modélisation issues de l'EDA

| Observation | Décision |
| --- | --- |
| Valeurs manquantes localisées | Imputation apprise sur le train (notebook 03) |
| Échelles hétérogènes | Scaling numérique obligatoire |
| Outliers légitimes | Winsorising plutôt que suppression |
| Modalités rares | Regroupement `rare` avant encodage |
| Colinéarité | Surveiller l'importance des features (notebook 04) |

**Suite** : `02_validation.ipynb` transforme ces observations en **contrats exécutables** (Pandera).